# Retry graphs still missing after notebook 03

Notebook 03 recovered 988 Devign and 4 REVEAL graphs. This notebook retries only its `still_missing` rows. It supplies additional documented parse-only macro stubs and uses `.cpp` for likely C++ signatures after a `.c` failure. The result is a separate, resumable journal. Recovered graphs require review because empty stubs for `STATUS_PARAM`/`TSRMLS_*` can omit contextual parameters. No CPG input or prompt bundle is overwritten.

Run in the project's WSL `.venv-wsl` kernel. A full run may take hours.


In [1]:
import json, sys, os, re, hashlib, tempfile
from pathlib import Path
from collections import Counter
ROOT=Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified')
sys.path.insert(0,str(ROOT))
from prepare_joern_cpg import extract_batch,run_joern_parse,run_joern_export,read_graphml,source_function_graph
OLD=ROOT/'output/local_full/joern_macro_recovery_20261031'
OUT=ROOT/'output/local_full/joern_second_recovery_20261031'
OUT.mkdir(parents=True,exist_ok=True)
JOERN=Path('/home/yungn/tools/joern/y/joern-cli')
JAVA_OPTS=os.environ.get('JOERN_JAVA_OPTS','-Xms128m -Xmx1536m')
BATCH_SIZE=10
assert JOERN.joinpath('joern-parse').is_file()
print('Joern:',JOERN,'Results:',OUT)


Joern: /home/yungn/tools/joern/y/joern-cli Results: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/joern_second_recovery_20261031


## Parse-only definitions

`_U_`, `U_CALLCONV` and `PNGAPI` are attribute/calling-convention markers in the inspected examples. `OPPROTO`, `STATUS_PARAM`, `STATUS_VAR`, `TSRMLS_DC` and `TSRMLS_CC` can depend on build configuration; empty definitions here are diagnostic approximations, not faithful project expansion. Every successful graph is marked `recovered_review`, never applied automatically. The original function text and both prelude and source hashes are recorded.


In [2]:
OLD_MACROS=('av_cold','coroutine_fn','always_inline','av_always_inline','av_noinline',
            'QEMU_NORETURN','attribute_align_arg','av_unused','CUDAAPI','WINAPI',
            'G_GNUC_UNUSED','av_flatten')
NEW_MACROS=('OPPROTO','STATUS_PARAM','STATUS_VAR','_U_','U_CALLCONV','PNGAPI',
            'TSRMLS_DC','TSRMLS_CC')
MACROS=OLD_MACROS+NEW_MACROS
PRELUDE=''.join(f'#ifndef {name}\n#define {name}\n#endif\n' for name in MACROS)
def digest(s): return hashlib.sha256(s.encode('utf-8')).hexdigest()
def load_rows(source):
    with source.open(encoding='utf-8') as f:
        for line in f:
            if line.strip(): yield json.loads(line)
def prior_missing(dataset,split):
    path=OLD/f'{dataset}_{split}.results.jsonl'
    if not path.exists(): raise FileNotFoundError(path)
    return [r for r in load_rows(path) if r['status']=='still_missing']
def likely_cpp(func):
    head=func.split('{',1)[0]
    return bool(re.search(r'\b(?:template|class|namespace)\b|::|\b[A-Za-z_]\w*\s*&\s*[A-Za-z_]\w*',head))
print({f'{d}/{s}':len(prior_missing(d,s)) for d in ('devign','reveal') for s in ('train','validation','test')})


{'devign/train': 111, 'devign/validation': 10, 'devign/test': 12, 'reveal/train': 2018, 'reveal/validation': 262, 'reveal/test': 232}


## Retry and checkpoint

Results are flushed after every batch. If the notebook stops, rerunning it skips IDs already journaled. A `still_missing` row with an error needs further investigation; some snippets are truncated or contain invalid preprocessor fragments. The `.cpp` fallback runs only for signatures with clear C++ markers.


In [3]:
def parse_cpp(func):
    with tempfile.TemporaryDirectory(prefix='grace-retry-cpp-') as tmp:
        root=Path(tmp); src=root/'src'; src.mkdir()
        (src/'sample.cpp').write_text(PRELUDE+func,encoding='utf-8')
        env=os.environ.copy(); env['JAVA_OPTS']=JAVA_OPTS
        run_joern_parse(src,root/'target.bin',JOERN,env,JAVA_OPTS)
        run_joern_export(root/'target.bin',root/'export',JOERN,env)
        graphs=list((root/'export').glob('*.xml'))+list((root/'export').glob('*.graphml'))
        if len(graphs)!=1: raise ValueError(f'Expected 1 GraphML, got {len(graphs)}')
        nodes,edges=read_graphml(graphs[0])
        return source_function_graph(nodes,edges)
def retry_split(dataset,split):
    source=prior_missing(dataset,split)
    journal=OUT/f'{dataset}_{split}.results.jsonl'
    done={r['id'] for r in load_rows(journal)} if journal.exists() else set()
    pending=[r for r in source if r['id'] not in done]
    print(dataset,split,'prior_missing',len(source),'done',len(done),'pending',len(pending),flush=True)
    with journal.open('a',encoding='utf-8',newline='\n') as f:
        for offset in range(0,len(pending),BATCH_SIZE):
            group=pending[offset:offset+BATCH_SIZE]
            batch=[(i,{'id':r['id'],'func':PRELUDE+r['original_func'],'target':0}) for i,r in enumerate(group)]
            try: results=extract_batch(batch,JOERN,JAVA_OPTS)
            except Exception as exc:
                results=[(i,row,{'nodes':[],'edges':[]},f'{type(exc).__name__}: {str(exc)[:500]}') for i,row in batch]
            for i,row,graph,error in results:
                original=group[i]['original_func']; mode='c'
                nodes,edges=graph.get('nodes') or [],graph.get('edges') or []
                if not nodes and likely_cpp(original):
                    try: nodes,edges=parse_cpp(original); error=None; mode='cpp'
                    except Exception as exc: error=f'{error}; cpp: {type(exc).__name__}: {str(exc)[:300]}'
                result={'id':row['id'],'dataset':dataset,'split':split,
                        'status':'recovered_review' if nodes else 'still_missing',
                        'mode':mode,'error':error,'original_func':original,
                        'func_sha256':digest(original),'prelude_sha256':digest(PRELUDE),
                        'parse_macros':list(MACROS),'nodes':nodes,'edges':edges,
                        'method_names':[n.get('name') for n in nodes if n.get('type')=='METHOD']}
                f.write(json.dumps(result,ensure_ascii=False)+'\n')
            f.flush(); os.fsync(f.fileno())
            print(dataset,split,min(offset+BATCH_SIZE,len(pending)),'/',len(pending),flush=True)
    return journal
journals=[retry_split(d,s) for d in ('devign','reveal') for s in ('train','validation','test')]


devign train prior_missing 111 done 0 pending 111
devign train 10 / 111
devign train 20 / 111
devign train 30 / 111
devign train 40 / 111
devign train 50 / 111
devign train 60 / 111
devign train 70 / 111
devign train 80 / 111
devign train 90 / 111
devign train 100 / 111
devign train 110 / 111
devign train 111 / 111
devign validation prior_missing 10 done 0 pending 10
devign validation 10 / 10
devign test prior_missing 12 done 0 pending 12
devign test 10 / 12
devign test 12 / 12
reveal train prior_missing 2018 done 0 pending 2018
reveal train 10 / 2018
reveal train 20 / 2018
reveal train 30 / 2018
reveal train 40 / 2018
reveal train 50 / 2018
reveal train 60 / 2018
reveal train 70 / 2018
reveal train 80 / 2018
reveal train 90 / 2018
reveal train 100 / 2018
reveal train 110 / 2018
reveal train 120 / 2018
reveal train 130 / 2018
reveal train 140 / 2018
reveal train 150 / 2018
reveal train 160 / 2018
reveal train 170 / 2018
reveal train 180 / 2018
reveal train 190 / 2018
reveal train 200 /

In [4]:
for path in journals:
    print(path.name,dict(Counter(r['status'] for r in load_rows(path))),flush=True)
print('Review method_names and code coverage in recovered_review rows before merging.')


devign_train.results.jsonl {'recovered_review': 94, 'still_missing': 17}
devign_validation.results.jsonl {'recovered_review': 6, 'still_missing': 4}
devign_test.results.jsonl {'recovered_review': 9, 'still_missing': 3}
reveal_train.results.jsonl {'recovered_review': 1621, 'still_missing': 397}
reveal_validation.results.jsonl {'recovered_review': 214, 'still_missing': 48}
reveal_test.results.jsonl {'recovered_review': 188, 'still_missing': 44}
Review method_names and code coverage in recovered_review rows before merging.
